# 06 — LLM-as-Judge: When and Why

This notebook does not run experiments. It documents when LLM-based evaluation adds value over standard IR metrics, and when it does not — so you can make an informed call before adding it to a pipeline.

## What LLM-as-judge is

An LLM is prompted to score each retrieved passage for relevance to a query, acting as a surrogate annotator. Tools like [RAGAS](https://docs.ragas.io/) wrap this into metrics such as `context_precision` (what fraction of top-k passages does the LLM consider relevant?) and `context_recall` (what fraction of the answer's required evidence was retrieved?).

Unlike NDCG/MRR, it does not require pre-existing qrels — it can be applied on any query/corpus combination.

## When standard IR metrics are sufficient

If you have a benchmark dataset with human-annotated relevance judgements (qrels) — BEIR, MS MARCO, NQ — NDCG and MRR are the right metrics. They are:

- **Reproducible:** same inputs always produce the same score
- **Fast and cheap:** no model inference on every query-passage pair
- **Interpretable:** decades of IR literature give clear baselines

This is the case for notebooks 01–05. NDCG@10 is the right tool there.

## Where LLM-as-judge earns its cost

**1. No ground-truth labels exist**  
You have a private corpus or a domain not covered by BEIR. There are no qrels. An LLM judge is the only automated path to a relevance signal without expensive human annotation.

**2. Qrel coverage is sparse (annotation gaps)**  
Large corpora are typically judged with pooling — only passages surfaced by some baseline retriever get annotated. A better retriever may find genuinely relevant passages that were never judged; these are penalised as false positives by NDCG even though they are correct. LLM judgment can surface these cases.

**3. Evaluating a full RAG pipeline, not just retrieval**  
Once a generator is in the loop, the question shifts from "did the right chunk rank first?" to "did the answer use the retrieved evidence correctly?". RAGAS `faithfulness` and `answer_relevancy` require an LLM to score — there is no qrel-based equivalent.

**4. Held-out queries with no reference answers (fine-tuning eval)**  
Notebook 07 fine-tunes a bi-encoder on domain data. If the held-out test queries have no annotated qrels, LLM judgment provides the only automated relevance signal for measuring improvement.

## Key caveats

**The judge must be capable for the domain.**  
A 3B parameter model judging SciFact (biomedical claims) will produce unreliable scores — not because LLM-as-judge is broken, but because the model lacks domain knowledge to assess relevance. For specialist domains, you need a larger or domain-adapted model. A small local model is fine for general-purpose or open-domain queries.

**Scores are not comparable across judge models.**  
Switching the judge (e.g. llama3.2:3b → GPT-4o) changes the absolute scores. Only use LLM judge scores for relative comparisons within the same judge setup.

**Cost scales with corpus × queries.**  
Each (query, passage) pair requires one LLM call. Budget accordingly, or sample aggressively (100–200 queries is usually enough for a directional signal).

## Summary

| Situation | Use LLM judge? |
|---|---|
| Benchmarked dataset with qrels (BEIR, MS MARCO) | No — NDCG/MRR is faster and authoritative |
| Private corpus, no labels | Yes — only automated option |
| Sparse qrel coverage, large corpus | Yes — catches annotation gaps |
| Evaluating a RAG pipeline end-to-end | Yes — faithfulness/answer quality need it |
| Fine-tuned model on domain data, no held-out qrels | Yes — use as a proxy signal |
| Comparing retrieval methods on known benchmarks | No — adds noise, not signal |

For this study, notebooks 01–05 use NDCG/MRR throughout. LLM-as-judge would apply in notebook 07 if the fine-tuned model is evaluated on held-out queries without annotated qrels.